# Variable Selection and First Nowcast

This notebook escalates in complexity.

This activity will use python to perform variable selection in a dataset and estimate a simple Nowcasting model based on OLS. We will:

- Load a full data
- Estimate LARS and compare it with OLS
- Nowcast GDP with OLS and several information sets

*Make sure the data folder is in the same directory as this notebook.*

This Notebook has **parts** of the code you need to to the tasks above. Follow the notes to complete or write the pieces of code you may need to finish.



## The config loader

This cell is the same in every notebook that uses `config.py`. It:

1. finds `config.py`;
2. loads it as a module called `cfg`, so every setting is available as `cfg.COUNTRY`,
   `cfg.TRAIN_START`, and so on;
3. derives the standard folder paths into `P`;
4. imports the model library from `nwcst_helpers.py`, which keeps this notebook thin.

Read the banner it prints. Everything in it came from `config.py`.

> **You edit `config.py`. You do not edit the notebook.**

In [ ]:
# ===========================================================================
# CONFIG LOADER: edit config.py, NOT this cell.
# Gives you `cfg` (every country knob) and `P` (the standard paths).
# ===========================================================================
import os, sys, importlib, importlib.util

# config.py and nwcst_helpers.py live in activity/, beside this notebook.
# From activity/ that is simply the current folder; from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)     # creates raw/, data/, output/, logs/ if missing

# ---------------------------------------------------------------------------
# The model library. joblib/loky workers re-import the module, so its folder
# must be importable, hence the sys.path insert.
# ---------------------------------------------------------------------------
for _p in (os.getcwd(), CODE_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import nwcst_helpers
importlib.reload(nwcst_helpers)    # pick up edits without restarting the kernel
from nwcst_helpers import *        # estimate_factors, evaluate_selection,
                                   # lars_variable_ranking, rank_sis/tstat/shap/ocmt/ibma,
                                   # deduplicate_correlated, gen_lagged_data, flatten_data,
                                   # mean_fill_dataset, lagged_target_q, fit_ols_flat,
                                   # predict_ols_flat, perform_tab, forecast_table

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}) | "
      f"train {cfg.TRAIN_START} -> test {cfg.TEST_START} -> target {cfg.DESIRED_DATE}")

## Libraries and settings

The star import above already brought in pandas, numpy and the scikit-learn pieces the
helper functions need. We import them again by name here so it is obvious what this
notebook itself uses.

Everything below the imports comes from `config.py` - the train/test window, the GDP
ticker, the CPI used to deflate, the number of PCA factors, the vintages. Change a
country there and this cell is the only thing that reacts.

In [ ]:
import warnings
warnings.filterwarnings('ignore')            # suppress non-critical warnings to keep output clean

import pandas as pd                          # pandas: the main library for tabular data (DataFrames)
import numpy as np                           # numpy: fast math and array operations
import matplotlib.pyplot as plt              # pyplot: plotting library (like ggplot in R)

from sklearn.linear_model import LinearRegression   # OLS regression from scikit-learn
from sklearn.decomposition import PCA               # Principal Component Analysis
from sklearn.preprocessing import StandardScaler    # standardize features to mean=0, std=1
from statsmodels.tsa.seasonal import seasonal_decompose  # trend + seasonal + residual

# ---------------------------------------------------------
# 1) DEFINE TRAIN / TEST WINDOW               (config.py)
# ---------------------------------------------------------
# - Training sample:   [train_start_date, test_start_date)
# - Test sample:       [test_start_date, test_end_date]
train_start_date = pd.to_datetime(cfg.TRAIN_START)
test_start_date  = pd.to_datetime(cfg.TEST_START)

# ---------------------------------------------------------
# 2) OUT-OF-SAMPLE FORECAST TARGET (LAST DATE / VINTAGE)
# ---------------------------------------------------------
desired_date = pd.to_datetime(cfg.DESIRED_DATE)

# ---------------------------------------------------------
# 3) SERIES AND MODEL SETTINGS
# ---------------------------------------------------------
# `from nwcst_helpers import *` also imported the module's own defaults for
# target_variable and scaler. Re-assert them here so the notebook's values win.
target_variable = cfg.GDP_TICKER   # the column name for real GDP (our Y variable)
cpi             = cfg.CPI_TICKER   # CPI series used to deflate nominal variables
scaler          = StandardScaler() # created once, re-fit wherever it is used

factors = cfg.LARS_FACTORS         # number of PCA factors used throughout the notebook

# Information-set vintages:
# negative = data available that many months BEFORE the GDP reference quarter
# positive = data available that many months AFTER it
lags = list(cfg.LAGS)

print(f"target {target_variable} | deflator {cpi} (base {cfg.CPI_BASE_YEAR}) | "
      f"factors {factors} | vintages {lags}")

# Load the data

In [ ]:
# pd.read_csv() loads a CSV file into a DataFrame; parse_dates=True converts date strings automatically
# index_col='date' sets the 'date' column as the row index instead of a plain column
data = pd.read_csv(os.path.join(P["data"], cfg.DATA_CSV),
                   parse_dates=True, index_col='date')

# .dropna() removes rows where GDP is NaN; .index.max() returns the latest date remaining
test_end_date = data[target_variable].dropna().index.max()

data.head(2)   # .head(n) shows the first n rows, useful for a quick sanity check

**COMPLEX SEASONALITY AND GROWTH RATES**

Our dataset has mixed frequencies. This time we will start accounting for mixed frequencies.

We are going to apply seasonality filters based on the data frequency. To do so, load the ```metadata.csv``` --a file that describes all variables in the data.

In [ ]:
metadata = pd.read_csv(os.path.join(P["data"], cfg.METADATA_CSV))   # series metadata
# The column with variable names
meta_col = 'ticket'   # 'ticket' is the column in metadata that holds the variable name/code

metadata   # displaying a variable on the last line of a cell prints it in the notebook output

The metadata describes which variables are real vs nominal. Use this information to deflate.

In [ ]:
# =========================================================
# Deflate nominal variables using CPI (base year: cfg.CPI_BASE_YEAR average, CBTT, Jan 2015 = 100)
# =========================================================
# Many series are nominal; deflating them puts everything in real terms.
# We compute a CPI base as the average CPI level in the base year.
base = data[cpi][data.index.year == cfg.CPI_BASE_YEAR].mean()
# data.index.year == 2019 is a boolean mask; it selects only rows where the year is 2019

# metadata.loc[condition][column].values extracts matching column values as a numpy array
nominal = metadata.loc[metadata["nominal"] == 1][meta_col].values

# Deflate: Real_t = Nominal_t / CPI_t * CPI_base   (corrected: was multiplying)
for column in data.columns:       # loop over every column name in the DataFrame
    if column in nominal:         # 'in' checks whether a value exists in a list/array
        data[column] = data[column] / data[cpi] * base    # beta11: divide (deflate) not multiply

# .drop() removes columns (axis=1) or rows (axis=0); inplace=True modifies the DataFrame directly
data.drop(columns=[cpi], inplace=True)
# Drop GDP components
# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
# list comprehension: builds a list of column names where 'RGDP' appears but keeping the target

Here we address seasonality with different filters depending on whether data is monthly, weekly, daily, or quarterly.

In [ ]:
# .str accessor lets you use string methods on a Series; .str.lower() converts to lowercase
quarterly  = metadata.loc[metadata["freq"].str.lower().str.contains("q")][meta_col].values
monthly    = metadata.loc[metadata["freq"].str.lower().str.contains("m")][meta_col].values
already_sa = metadata.loc[metadata["sa"]==1][meta_col].values   # sa=1 means already seasonally adjusted

# Keep only columns with at least 24 non-NaN values; .notna().sum() counts non-missing entries per column
data = data.loc[:, data.notna().sum() >= 24]

for column in data.columns:
    if column in already_sa:
        continue   # 'continue' skips the rest of the loop body and moves to the next column
    series = data[column].dropna()   # remove NaN before decomposing (seasonal_decompose requires it)
    if column in quarterly:
        # period=4: quarterly series has 4 seasons per year
        decomp = seasonal_decompose(series, model='additive', extrapolate_trend='freq', period=4)
        data[column] = data[column] - decomp.seasonal   # subtract seasonal component to remove seasonality
    else :
        # period=12: monthly series has 12 seasons per year
        decomp = seasonal_decompose(series, model='additive', extrapolate_trend='freq', period=12)
        data[column] = data[column] - decomp.seasonal


Now calculate growth rates:

In [ ]:
# =========================================================
# Convert levels to growth rates (percent change)
# =========================================================
# Quarterly series: quarter-over-quarter growth rate
# All other series: default percent change at their observed frequency
for column in data.columns:
    if column in quarterly:
        # Get non-NaN values with their original index
        non_nan = data[column].dropna()
        
        # .pct_change(periods=1) computes (x_t - x_{t-1}) / x_{t-1}; * 100 gives percentage
        pct_change_values = non_nan.pct_change(periods=1, fill_method=None) * 100
        
        # Create a full-length Series of NaN, then fill in only the computed positions
        result = pd.Series(np.nan, index=data.index)
        result.loc[pct_change_values.index] = pct_change_values   # .loc with an index selects by label
        
        data[column] = result
    else:
        # Store original NaN mask
        original_was_nan = data[column].isna()   # boolean Series: True where value is missing
        
        # For monthly/weekly/daily, calculate normally
        pct_change_values = data[column].pct_change(periods=1, fill_method=None) * 100
        
        # Mask out cases where either current or previous value was originally NaN
        prev_was_nan = data[column].shift(1).isna()   # .shift(1) shifts values down by 1 row
        pct_change_values[prev_was_nan | original_was_nan] = np.nan   # | is the bitwise OR for booleans

        data[column] = pct_change_values

# np.inf arises from pct_change when dividing by zero; replace with NaN to keep data clean
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================================================
# Drop columns in quarterly with more than 75% missing values
# =========================================================
for column in [ col for col in data.columns if col not in quarterly ] :
    # compute fraction of missing values in the training period only
    missing_pct = data[data.index<=test_start_date][column].isna().sum() / len(data[data.index<=test_start_date][column])
    if missing_pct > 0.75:
        data.drop(column, axis=1, inplace=True)

# =========================================================
# Final cleaning
# =========================================================
# Replace infinities created by pct_change with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop CPI from predictor set (used only for deflation)
#data.drop(cpi, axis=1, inplace=True)

# ~ negates a boolean Series; data[condition] filters rows where condition is True
data = data[~(data.index < train_start_date)].reset_index(drop=False)
# .reset_index(drop=False) moves the date index back to a regular column
data = data.set_index("date").dropna(how="all", axis=0)   # drop rows that are entirely NaN

# pd.to_numeric(..., errors='coerce') converts to number; non-numeric values become NaN
metadata["months_lag"] = (
    pd.to_numeric(metadata["months_lag"], errors="coerce")
      .fillna(1)      # replace any NaN lags with 1 (safe default)
      .astype(int)    # .astype(int) casts the Series to integer dtype
)

# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
for col in data.columns :
    last = data[col].dropna().index.max().year   # find the most recent year with data for this column
    if last < 2024 :
        data.drop(col, axis=1, inplace=True)   # remove columns that stopped updating before 2024

Remove highly correlated predictors before the selection race. Sub-components of the
same category (e.g. BOP financial-account sub-accounts) are nearly collinear - a selector
picks several of them without adding information. For each correlated pair
`deduplicate_correlated()` keeps whichever variable is an aggregate (ticket number ends in
`00`); otherwise it keeps the one with more non-null observations.

The function is in `nwcst_helpers.py`. Open it if you want to see the rule it applies.

In [ ]:
data = deduplicate_correlated(data, threshold=0.98,
                              target=target_variable,
                              metadata=metadata, meta_col=meta_col)

# Drop high-publication-lag series: anything published more than 5 months late is
# useless for a nowcast, however well it correlates.
tickets_to_keep = metadata[metadata.months_lag <= 5]['ticket'].tolist()

# Keep only these columns (that exist in data)
data = data[[ col for col in data.columns if col in tickets_to_keep or "RGDP" in col ]]

Glimpse at the data

In [ ]:
# =========================================================
# Data glimpse: GDP plus a few high-frequency indicators.
# Any ticker that did not survive the cleaning above (or does not exist for
# another COUNTRY) is simply skipped, so switching country degrades the figure
# instead of raising KeyError.
# =========================================================
WANTED = [
    (target_variable, 'Real GDP Growth (QoQ %)',       'GDP',                        '#003865', 'bar'),
    ('RENE5005',      'LNG Production Growth (MoM %)', 'LNG production (RENE5005)',  '#009CA7', 'line'),
    ('UGTR0000',      'Google Trends Growth (MoM %)',  'Google Trends (UGTR0000)',   '#F5A623', 'line'),
    ('UNTL0000',      'Nighttime Lights Growth (MoM %)','Nighttime Lights (UNTL0000)','#8B5CF6', 'line'),
]
panels  = [p for p in WANTED if p[0] in data.columns]
missing = [p[0] for p in WANTED if p[0] not in data.columns]
if missing:
    print(f"Not plotted (absent from data): {', '.join(missing)}")

fig, axes = plt.subplots(len(panels), 1, figsize=(13, 3.25 * len(panels)), sharex=False)
axes = np.atleast_1d(axes)   # keeps the loop working when there is only one panel

for ax, (ticker, title, label, color, kind) in zip(axes, panels):
    s = data[ticker].dropna()
    if kind == 'bar':
        ax.bar(s.index, s, width=60, color=color, alpha=0.8, label=label)
    else:
        ax.plot(s.index, s, color=color, linewidth=1.8, label=label)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(title, fontsize=11, weight='bold')
    ax.set_ylabel('% change')
    ax.legend(frameon=False)
    ax.grid(axis='y', linestyle=':', alpha=0.5)

fig.suptitle('Data Glimpse: Selected Indicators', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Reproducibility: print core library versions
# =========================================================
print("================== Library Versions =================")
import sklearn, numpy, pandas
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)


# Data dimensions (observations and variables)
print("Data dimensions:", data.shape)
print(f"Training dates: {train_start_date.strftime('%Y-%m-%d')} to {(test_start_date - pd.DateOffset(days=1)).strftime('%Y-%m-%d')}")
T1 = data[data.index < test_start_date].shape[0]
T2 = data[(data.index >= test_start_date) & (data.index <= test_end_date)].shape[0]
print(f"Training periods: {T1} ({T1/(T1+T2) :.2%})")
print(f"Testing dates: {test_start_date.strftime('%Y-%m-%d')} to {test_end_date.strftime('%Y-%m-%d')}")
print(f"Testing periods: {T2} ({T2/(T1+T2) :.2%})")

# Variable Selection: LARS


Instead of trusting one selection method, we let several of them **compete on
equal terms**, following Chinn, Meunier & Stumpner (2023):

1. every method ranks the candidate variables **once**, using the training
   window only;
2. each ranking is cut at every size (10, 11, ... variables);
3. every cut is scored by the **same frozen evaluator** (PCA + OLS,
   out-of-sample RMSE) against the **same benchmark** (all variables).

Because steps 2 and 3 are identical for everyone, any difference in RMSE is
caused by the *selector* alone.

However, for now we use only the LARS selection method

### Where the functions live

Nothing below defines a function. Everything the race needs was imported from
`nwcst_helpers.py` by the config-loader cell:

| Function | What it does |
|---|---|
| `estimate_factors` | Bai & Ng (2002) information criteria for the number of PCA factors |
| `evaluate_selection` | **the frozen evaluator**: standardize -> PCA -> OLS -> out-of-sample RMSE. Identical for every method, fitted on train only |
| `lars_variable_ranking` | Least Angle Regression; with `test=None` it only ranks |
| `rank_sis` | Sure Independence Screening - absolute marginal correlation (Fan & Lv 2008) |
| `rank_tstat` | Univariate screens controlling for GDP's own lags, HAC errors (Bai & Ng 2008) |
| `rank_shap` | Gradient Boosting + mean abs. SHAP. The only non-linear competitor; needs `pip install shap` |
| `rank_ocmt` | One Covariate at a Time Multiple Testing (Chudik, Kapetanios & Pesaran 2018) - brings its own threshold |
| `rank_ibma` | Iterated Bayesian Model Averaging, BIC weights (Yeung et al. 2005) |

Open `nwcst_helpers.py` and read them. Each `rank_*` takes the training window and
returns an **ordered list of variables**, best first. All of the rankings are *nested*:
the top-20 is simply the first 20 entries of the top-50, which is what lets us rank once
and slice.

Factors are defined in the config file

In [ ]:
print(cfg.LARS_FACTORS)

### Select with the data we will actually have

LARS ranks the variables on quarterly values. If we give it each variable's value for the **same**
quarter as GDP, it assumes everything is published on time. It is not: most series here come out
3 or 4 months after the month they describe, so when we nowcast, the target quarter is mostly
still empty. LARS would then choose variables for a co-movement the nowcast cannot see.

So we build the selection data the way it looks at **vintage -2**, two months before the end of
the quarter. `gen_aligned_data()` (in `nwcst_helpers.py`) shifts each series by its publication
lag plus 2 months: the value in a given month is the latest one already published at that point.
GDP, the target, is not shifted.

In [ ]:
SELECT_LAG = -2   # the vintage used for selection: 2 months before the end of the quarter

aligned = gen_aligned_data(metadata, data.reset_index(), data.index.max(), SELECT_LAG)
aligned = aligned.set_index("date")

# One series, before and after: the aligned column is the published one, moved down
ex = [c for c in data.columns if c != target_variable][0]
print(ex, "| publication lag:", metadata.loc[metadata.ticket == ex, "months_lag"].values[0], "months")
pd.concat([data[ex], aligned[ex]], axis=1, keys=["published", "known at vintage -2"]).loc["2025"]

After the functions are prepared, we proceed by reshaping our dataset.

The data has mixed frequencies. We will aggregate data into quarterly by using the average, sum, and end-of-period values. We define in the metadata what type of aggregation to use for each variable.

In [ ]:
# .set_index('ticket') makes ticket the index so we can look up agg_method by variable name
# .to_dict() converts the resulting Series into a plain Python dictionary {ticket: agg_method}
agg_map = metadata.set_index('ticket')['agg_method'].to_dict()

# Dictionary comprehension: build {col: aggregation_function} for every column in data
# If agg_method is not 'mean', 'last', or 'sum', fall back to 'mean'
agg_funcs = {
    col: (agg_map[col] if agg_map[col] in ['mean', 'last', 'sum'] else 'mean')
    for col in data.columns if col in agg_map
}
# .resample('QS') groups observations by quarter-start; .agg() applies a different function per column
pre_selected = aligned.resample('QS').agg(agg_funcs)   # the vintage -2 data, not the final data
pre_selected.tail(5)   # .tail(5) shows the last 5 rows

Some additional data management:

1. Make sure the index is a date.
2. Drop missing values (over 75% missing > then dropped)
3. Fill missing values with the mean

In [ ]:
pre_selected.index = pd.to_datetime(pre_selected.index)   # ensure the index is a proper DatetimeIndex
pre_selected = pre_selected.dropna(how="all", axis=1)     # drop columns that are entirely NaN
# .isnull().mean() gives the fraction of missing values per column; keep columns with <= 75% missing
pre_selected = pre_selected.loc[:, pre_selected.isnull().mean() <= 0.75]

# Mean-fill predictors only; leave target_variable as NaN so we can
# identify quarters without observed GDP and drop them from training.
# List comprehension: all columns except the GDP target
predictors = [c for c in pre_selected.columns if c != target_variable]
# .fillna(mean) replaces NaN with the column mean; applied only to predictor columns
pre_selected[predictors] = pre_selected[predictors].fillna(
    pre_selected[predictors].mean(numeric_only=True)
)

pre_selected.tail(5)


Another cleaning step: data cannot include periods beyond observed GDP.

So we find the latest date for which GDP is available and drop observations beyond that date.

In [ ]:
last_gdp = data[target_variable].dropna()   # Series containing only the non-NaN GDP observations
last_gdp = last_gdp.index.max()             # .index.max() gives the latest date that has observed GDP
# Boolean indexing: keep only rows where the date is <= last observed GDP date
pre_selected = pre_selected[pre_selected.index <= last_gdp]
# dropna(subset=[...]) drops rows where the specified column is NaN (removes unobserved GDP quarters)
pre_selected.dropna(subset=[target_variable], inplace=True)
print(f"First date: {pre_selected.index.min()}")   # f-string: embeds variable values inside the string
print(f"Last date: {pre_selected.index.max()}")

### Benchmark

This approach uses OLS as a benchmark. The benchmark uses all variables in the dataset, reduced to
`factors` PCA factors (set by `LARS_FACTORS` in `config.py`; 1 here), to predict GDP growth and
estimate the RMSE.

Now estimate the factors and OLS.

In [ ]:
# Split raw data first
X_raw = pre_selected.drop(target_variable, axis=1)   # all predictor columns (drop GDP)
y_all = pre_selected[[target_variable]]               # double brackets [[ ]] keep it as a DataFrame (not a Series)

# Boolean indexing splits X and y into train and test portions by date
X_train_raw = X_raw[X_raw.index <  pd.to_datetime(test_start_date)]
X_test_raw  = X_raw[X_raw.index >= pd.to_datetime(test_start_date)]
y_train     = y_all[y_all.index <  pd.to_datetime(test_start_date)]
y_test      = y_all[y_all.index >= pd.to_datetime(test_start_date)]

# Drop quarters where GDP was not observed (NaN target = no real observation)
obs_mask    = y_train[target_variable].notna()   # boolean mask: True where GDP is available
y_train     = y_train[obs_mask]
X_train_raw = X_train_raw[obs_mask]

# scaler.fit_transform() learns mean & std from train, then standardizes; applied to train only
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw), index=X_train_raw.index)
# scaler.transform() applies the SAME scaling learned from train. Never refit on test data!
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_raw),      index=X_test_raw.index)

# pca.fit_transform() learns principal components from train, then projects; applied to train only
pca = PCA(n_components=factors)
X_train = pd.DataFrame(pca.fit_transform(X_train_scaled), index=X_train_raw.index)
X_test  = pd.DataFrame(pca.transform(X_test_scaled),      index=X_test_raw.index)
# Rename columns to PC_0, PC_1, ... using a list comprehension
X_train.columns = [f"PC_{i}" for i in range(factors)]
X_test.columns  = [f"PC_{i}" for i in range(factors)]

benchmark_model = LinearRegression()        # instantiate the OLS model object
benchmark_model.fit(X_train, y_train)       # .fit() estimates coefficients using training data
print(f"Training on {len(y_train)} observed GDP quarters")


Recover the predicted values on the test period and compute the benchmark RMSE

In [ ]:
# .predict() returns a 2-D array; .ravel() flattens it to a 1-D array
y_pred = benchmark_model.predict(X_test).ravel()
y_true = y_test.values.ravel()   # .values converts a DataFrame/Series to a plain numpy array

# Naive baseline: always predict the training-period mean
naive_pred = float(y_train[target_variable].mean())   # float() converts a numpy scalar to a Python float
naive_rmse = float(np.sqrt(np.mean((y_true - naive_pred) ** 2)))   # RMSE formula: sqrt(mean of squared errors)

benchmark = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))  # RMSE for OLS benchmark

print(f"OLS benchmark RMSE : {benchmark:.4f}")   # :.4f in f-string formats the number to 4 decimal places

### LARS Estimation

LARS is estimated with a maximum number of variables. How many variables should we allow in the model? We don't know...

For now, let's use 10.

In [ ]:
# Split the quarterly data into train and test using boolean indexing on the date index
pre_train = pre_selected[pre_selected.index <  test_start_date]
pre_test  = pre_selected[pre_selected.index >= test_start_date]

# lars_variable_ranking returns (ordered list of selected variable names, out-of-sample RMSE)
selection, rmse = lars_variable_ranking(pre_train, target=target_variable, max_vars=10, test=pre_test, factors=factors)

# Build a dictionary (key: value pairs) to store this specification's results
b = {
    "size" : 10 ,
    "selected_vars" : selection ,   # list of variable names selected by LARS
    "rmse" : rmse
}

# pd.DataFrame([b]) converts a list of one dictionary into a one-row DataFrame
spec_list = pd.DataFrame([b])

spec_list

We find that, when LARS has to select 10 variables, it select the following:

In [ ]:
# .idxmin() returns the index label of the row with the smallest 'rmse' value
top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars']

# .isin(top_vars) filters the metadata DataFrame to rows whose ticket is in the selected list
metadata[metadata['ticket'].isin(top_vars)]

The million dollar question is: what is the right dimension? Should we use 10
variables? 15 or 100?


Just as importantly: is LARS even the best way to choose them? We have a variety of alternative methodologies in the toolkit

Values below 1 mean the selection beats the
kitchen-sink OLS.


In [ ]:
d = []   # empty list; will collect one dictionary per model size
for size in range(10,50) :   # range(10, 50) generates integers 10, 11, ..., 49
    selection, rmse = lars_variable_ranking(pre_train, target=target_variable, max_vars=size+1, test = pre_test, factors = factors)
    b = {
        "size" : size+1 ,
        "selected_vars" : selection ,
        "rmse" : rmse
    }
    d.append(b)   # .append() adds one item to the end of the list
d = pd.DataFrame(d)   # convert the list of dicts into a DataFrame (each dict becomes a row)
d.set_index('size', inplace=True)   # use model size as the row index for easy plotting

d['ratio'] = d['rmse']/benchmark   # add a new column: ratio of LARS RMSE to the OLS benchmark RMSE

d.head(5)

We can visualize this:

In [ ]:
# Best threshold is 1: where models outperform non-selected OLS
threshold = 1
d['ratio'].plot(color='black')                        # pandas .plot() wraps matplotlib for quick line charts
plt.axhline(y=threshold, color='black', linestyle=':', linewidth=1)  # draw a horizontal reference line at y=1
plt.ylabel("Ratio of RMSE")
plt.xlabel("Number of variables")
plt.legend().set_visible(False)
plt.show()   # plt.show() renders the figure; without it, the plot may not display outside notebooks

print(f"OLS benchmark RMSE : {benchmark:.4f}")
print(f"LARS min RMSE : {d['rmse'].min():.4f}")   # .min() returns the smallest value in the column

We nowcast with **one** LARS specification: the size with the lowest RMSE in the race above.
Look at its `ratio`: below 1 means this LARS set beats the all-variables benchmark, above 1
means it is only the least bad LARS set.

**A warning about this choice: the test window is used twice.** The RMSE that ranked the sizes
was computed on the test quarters, and the nowcast below is scored on the same test quarters.
Choosing the size that did best on the test data and then reporting its test RMSE is look-ahead:
with 40 sizes to choose from and only about 11 test quarters, some size will look good partly by
luck, and choosing it takes advantage of that luck. The final RMSE is therefore a **best case**.

The honest version keeps three windows apart:

1. **estimate** on the training window;
2. **choose** the size on a validation window (for example, the last 8 quarters of the training
   period), never touching the test quarters;
3. **score** the chosen size once, on the test window.

We keep the shortcut to keep the notebook short. Keep it in mind when you read the RMSE at the end.

In [ ]:
# Keep the size with the lowest RMSE (the best-performing specification)
spec_list = d[d['rmse'] == d['rmse'].min()]
spec_list

## Basic Nowcasting

Nowcasting uses a variety of functions. These functions:

- Generate lagged data according to the information set
- Flatten the data into a wide format where additional columns represent periods of the quarter
- Fill missing values with the mean

The next cell defines them so you can read them. Four more come from `nwcst_helpers.py`:

| Function | What it does |
|---|---|
| `fit_factors` / `apply_factors` | fit the scaler and the PCA on the **training** rows, then project any other frame on those same loadings |
| `quarter_rows` | keep the quarter-end months (March, June, September, December) |
| `lagged_target_v` | GDP's own lags, built with GDP's publication delay at each vintage |
| `fit_ols_flat` / `predict_ols_flat` | the whole recipe for one vintage: flatten, quarter rows, GDP lags, OLS |

In [ ]:
# lags is a list of integers representing information-set vintages
# negative = data available that many months BEFORE the GDP reference quarter
# positive = data available that many months AFTER the GDP reference quarter
lags = list(range(-2, 3))   # range(-2, 3) gives -2, -1, 0, 1, 2; list() converts it to a list

# helper function, generate lagged datasets for testing on vintages
def gen_lagged_data(metadata, data, last_date, lag):
    # only go up to the last date
    lagged_data = data.loc[data.date <= last_date, :].reset_index(drop=True)
    for col in lagged_data.columns[1:]:   # [1:] slices the column list, skipping the first column (date)
        pub_lag = metadata.loc[metadata.ticket == col, "months_lag"].values[0] # publication lag of this particular variable
        # go back as far as needed for the pub_lag of the variable, then + the lag (so -2 for 2 months back), also -1 because 0 lag means in month, last month data available, not current month in
        lagged_data.loc[(len(lagged_data) - pub_lag + lag ) :, col] = np.nan
        # set the last pub_lag+lag rows to NaN â€” those values haven't been published yet

    return lagged_data

# helper function, flatten a dataset for methods that don't do timeseries, extra columns for each lag
def flatten_data(data, target_variable, n_lags):
    # keep only rows where GDP is observed (non-NaN)
    flattened_data = data.loc[~pd.isna(data[target_variable]), :]
    orig_index = flattened_data.index   # save the original row positions
    for i in range(1, n_lags):   # loop i = 1, 2, ..., n_lags-1
        lagged_indices = orig_index - i          # shift index back by i rows
        lagged_indices = lagged_indices[lagged_indices >= 0]   # drop any negative indices
        tmp = data.loc[lagged_indices, :].copy()   # .copy() avoids modifying the original DataFrame
        tmp['date'] = tmp['date'] + pd.DateOffset(months=i)   # DateOffset shifts dates forward by i months
        tmp = tmp.drop([target_variable], axis=1)
        # rename columns to indicate they are lagged (e.g., "VAR_1" = value 1 period ago)
        tmp.columns = [j + "_" + str(i) if j != "date" else j for j in tmp.columns]
        flattened_data = flattened_data.merge(tmp, how="left", on="date")   # left join on date column

    return flattened_data

    
# helper function fill missings in a dataset with the mean from the training set
def mean_fill_dataset(training, test):
    mean_dict = {}   # empty dict; will map column name -> training mean
    for col in training.columns[1:]:
        mean_dict[col] = np.nanmean(training[col])   # np.nanmean ignores NaN when computing the mean
    filled = test.copy()
    for col in training.columns[1:]:
        # replace NaN in test with the training mean (avoids data leakage from test set)
        filled.loc[pd.isna(filled[col]), col] = mean_dict[col]
        
    return filled

# fit_ols_flat(), predict_ols_flat() and lagged_target_v() come from nwcst_helpers.py


def perform_tab( pred_values , model_name , specification , values , lags=lags) :
    # table of RMSE by vintage
    table = pd.DataFrame(columns=[ "Vintage", "RMSE", "estimator", "spec" ])
    for lag in lags:
        tmp = pd.DataFrame({
            "Vintage" : lag,
            # np.array() converts lists to arrays so element-wise subtraction works correctly
            "RMSE" : np.sqrt(np.mean((np.array(values) - np.array(pred_values[lag])) ** 2)) ,
            "estimator": model_name ,
            "spec" : specification ,
        }, index=[0])   # index=[0] required when all values are scalars (not lists)
        table = pd.concat([table, tmp]).reset_index(drop=True)   # pd.concat stacks DataFrames vertically
    table.round(4)
    table.set_index('Vintage' , inplace=True)
    return table

def forecast_table( pred_values , model_name , specification, values  , dates ) :
    # plot of predictions vs actuals
    result = pd.DataFrame(
        {
            "observed": actuals ,
            "two_back": pred_values[-2] ,   # dictionary key -2: info set 2 months before GDP release
            "one_back": pred_values[-1] ,
            "zero_back": pred_values[0] ,   # key 0: contemporaneous information set
            "one_ahead": pred_values[1] ,
            "two_ahead": pred_values[2] ,
            "estimator": model_name ,
            "spec": specification ,
        }
    )
    result.index = pd.to_datetime(dates)   # assign proper date index to the result table
    return result


We are going to start the Nowcasting exercise with the original (cleaned) dataset in mixed frequencies (Not quarterly)

(The reason is that LARS cannot handle mixed frequencies and flattening. The nowcasting methodology can handle it).

However, limit the variable list to only those selected by LARS + GDP (the target variable)

In [ ]:
# .values[0] extracts the first (and only) element from the numpy array stored in the cell
XVars = spec_list['selected_vars'].values[0]
print(XVars)         # print the list of selected variable names
print(len(XVars))    # len() returns the number of items in a list

In [ ]:
# Keep only LARS-selected predictors plus the GDP target; list comprehension filters data.columns
df = data[[ col for col in data.columns if col in XVars or col == target_variable ]]
print(f"Data shape: {df.shape}")   # .shape returns (number of rows, number of columns)
df.tail(2)

**Generate Test dataset**

The test dataset is tricky. It is not a real test dataset. Sorry. Unfortunate name choice.

The test dataset includes all the train data + test data. This was done due to PCA construction.

We will separate it into an actual training and test dataset later.

In [ ]:
df = df.reset_index(drop=False)   # move the date index back to a plain column called 'date'
# .dropna(axis=1, thresh=23): drop any column that has fewer than 23 non-NaN values
df.dropna(axis=1, thresh=23, inplace=True)
# Filter rows within the full sample window (train + test)
test = df.loc[(df.date >= train_start_date) & (df.date <= test_end_date), :].reset_index(drop=True)
# DATES AND ACTUAL VALUES
# pd.date_range generates evenly spaced quarterly dates; freq="3MS" = every 3 month-starts
# .strftime("%Y-%m-%d") formats each Timestamp as a string; .tolist() converts to a plain Python list
dates = ( pd.date_range(test_start_date, test_end_date, freq="3MS").strftime("%Y-%m-%d").tolist() )
# .isin(dates) checks which rows in test fall on a test-period GDP date; .values extracts as array
actuals = list(test.loc[test.date.astype(str).isin(dates), target_variable].values)

The list 'dates' includes all (actual) test dates. For each date in the test, we are going to generate a nowcast. The nowcast depends on what month we are located because each month has a different information set.

What that means is that training models are rolling over time, benefitting from additional information that gets published.

In [ ]:
dates

## Estimation with one information set and one period

In [ ]:
# param_pca is the number of factors included.
param_pca = 1
# " ".join(list) concatenates all items in XVars into a single space-separated string
spec = " ".join(XVars)

date = dates[0]   # take the first test date (dates is a list of strings)
lag = -2          # the vintage: -2 means 2 months before the end of the GDP reference quarter
date, lag

The training set rolls over time, depending of the test period we are nowcasting. Therefore, it is defined as a function of the test period.

In [ ]:
# define train dataset
# pd.to_datetime(date) converts the string date to a Timestamp so we can do arithmetic
# DateOffset(months=3) shifts the date back by one quarter â€” training ends before the test quarter
train = test.loc[test.date <= str(pd.to_datetime(date) - pd.tseries.offsets.DateOffset(months=3))[:10],:]
# str(...)[:10] slices the first 10 characters of the Timestamp string to get "YYYY-MM-DD"
train

We populate missing values

In [ ]:
transformed_train = mean_fill_dataset(train, train)
transformed_train

Reduce dimensions with a PCA.

In [ ]:
# Fit the scaler and the PCA on the TRAINING rows only. fit_factors() keeps both,
# so the test rows can later be projected on exactly the same factors.
fac = fit_factors(transformed_train, param_pca)
transformed_train = apply_factors(transformed_train, fac)   # date + GDP + PC_0, PC_1, ...
transformed_train

The data has mixed frequencies: GDP is quarterly (although we observe the mean instead of missing values) but the other variables are monthly.

Data flattening adds the months of a quarter as additioanl columns.

In [ ]:
flats = 3   # number of within-quarter months to include as extra columns (months 1, 2, 3 of each quarter)
# flatten_data adds lagged columns so each quarterly row carries all monthly info for that quarter
transformed_train = flatten_data(transformed_train, target_variable, flats )
transformed_train

Now keep only quarters.

In [ ]:
# quarter_rows() keeps only quarter-end months (Mar, Jun, Sep, Dec)
# .fillna(mean) fills any remaining NaN with column means; .dropna(axis=1) drops all-NaN columns
transformed_train = quarter_rows(transformed_train)
transformed_train = transformed_train.fillna(transformed_train.mean(numeric_only=True)).dropna(axis=1)
transformed_train.tail(5)

We may also want GDP's own lags (autoregressive terms) in the model. **Careful:** GDP is
published with a long delay (`months_lag` in the metadata: 10 months for Trinidad and Tobago),
so in real time the latest GDP we know is several quarters old.

`lagged_target_v()` builds the lags with that delay for vintage `lag`. At vintage -2 the delay is
12 months, so `RGDP0000_l1` is GDP four quarters back. The **training rows use the same delay as
the prediction row**: the model never learns from a GDP value it would not have when nowcasting.
(Missing values are filled with the mean.)

In [ ]:
gdp_lags = 2   # include 2 autoregressive lags of GDP as additional predictors
# lagged_target_v() appends RGDP0000_l1, RGDP0000_l2, built with GDP's publication delay at this vintage
transformed_train = lagged_target_v(transformed_train, gdp_lags, metadata, lag)
transformed_train

Estimate the model

In [ ]:
model = LinearRegression()   # create a fresh OLS model object
# Build X (predictors) by dropping the date and GDP target columns
x = transformed_train.drop(["date", target_variable], axis=1)
y = transformed_train[target_variable]
model.fit(x, y)   # estimate OLS coefficients; the fitted model is returned (and stored in 'model')

With the trained model, it is time to work on the test sample.

The test sample includes observations that are unavailable due to publication lags. These publication lags are defined in the metadata.

We artificially set up the test data to include missings according to the publication lag.

In [ ]:
date

In [ ]:
# lag = -2 was set above: 2 months before the end of the GDP reference quarter
# gen_lagged_data() artificially sets future observations to NaN to simulate publication lags
tmp_data = gen_lagged_data(metadata, test, date, lag)
tmp_data

Let's have a look at publication lags and vintages in action

In [ ]:
# Publication Lags Example:
# Select an example column
tmp_data.columns

In [ ]:
ex_col = [c for c in XVars if c in test.columns][0]   # first LARS-selected series
test[['date', ex_col]].tail(10)

In [ ]:
# to see how gen_lagged_data() blanked out the most recent observations
tmp_data[['date', ex_col]].tail(10)

Remember we are currently in month:

In [ ]:
date, lag

In [ ]:
metadata[metadata['ticket'] == ex_col]

Fill missing values with the training means, then project on the factors fitted on the training
data. `apply_factors()` only projects: it does **not** re-fit the scaler or the PCA. Re-fitting them
on the test frame would give factors that can rotate, or flip sign, relative to the ones the model
was estimated on.

In [ ]:
# Fill NaN in test data using means computed from the training data (avoids leakage)
tmp_data = mean_fill_dataset(train, tmp_data)
# Same scaler and same PCA as the training rows
tmp_data = apply_factors(tmp_data, fac)
tmp_data

To predict in the test set using coefficients from the training model, we need to emulate the same columns as in the training sample.
Therefore, follow the same procedure as before.

In [ ]:
# Apply the same flattening, quarter rows and GDP lags as in training
tmp_data = flatten_data(tmp_data, target_variable, flats )
tmp_data = quarter_rows(tmp_data)
tmp_data = lagged_target_v(tmp_data, gdp_lags, metadata, lag)
# Keep only the single row for the target date; drop date and GDP (not needed for prediction)
tmp_data = tmp_data.loc[tmp_data['date']== date, :].drop(["date", target_variable], axis=1)
tmp_data = tmp_data[x.columns]   # same columns, same order as in training
tmp_data

Now get the predicted value for a test date, under an information set lag.

In [ ]:
# model.predict() returns an array; [0] pulls out the scalar nowcast value
pred = model.predict(tmp_data)[0]
# Displaying a tuple prints all three values side by side in the notebook output
date, lag, pred

## Estimation and Prediction Loop

The estimation uses `param_pca` factors (1 here) of the selected variables.

For every test date, the training window grows and the factors are fitted on it. For every
vintage the OLS is re-fitted, because the GDP lags it can use depend on the vintage.

```transformed_train``` is a dataset that includes the selected variables as factors without missing values.

In [ ]:
# param_pca is the number of factors included.
param_pca = 1
# " ".join(list) builds a single string from list items, used as a readable label for this spec
spec = " ".join(XVars)

# Dictionary comprehension: create {lag: []} for each lag in lags
# This will store predicted GDP values for each information-set vintage
pred_ols = { k: [] for k in lags }
# Empty DataFrames that will be filled by concatenating results in the loop
performance_table = pd.DataFrame()
result_table = pd.DataFrame()

for date in dates :    # outer loop: iterate over each quarterly test period

    # Rolling training set: everything before the current test quarter
    train = test.loc[test.date <= str(pd.to_datetime(date) - pd.tseries.offsets.DateOffset(months=3))[:10],:]
    transformed_train = mean_fill_dataset(train, train)   # fill NaN with training means

    # Factors fitted on this training window only, then reused for the test rows
    fac = fit_factors(transformed_train, param_pca)
    train_f = apply_factors(transformed_train, fac)

    for lag in lags:   # inner loop: iterate over each vintage (information set)
        # Re-fit for each vintage: the GDP lags the model may use depend on the vintage
        ols_fit, ols_vars = fit_ols_flat(train_f, metadata, lag,
                                         gdp_lags = gdp_lags, flat_lags = flats)

        # the data available for this date at this artificial vintage
        tmp_data = gen_lagged_data(metadata, test, date, lag)
        tmp_data = mean_fill_dataset(train, tmp_data)
        tmp_data = apply_factors(tmp_data, fac)          # same scaler and PCA as training

        # Predicted values: append nowcast for this (date, lag) to pred_ols
        pred_ols = predict_ols_flat(ols_fit, tmp_data, ols_vars, date, pred_ols, lag, metadata,
                                    flat_lags = flats, gdp_lags = gdp_lags)

# pd.concat stacks the per-specification tables vertically into one summary table
performance_table = pd.concat( [ performance_table , perform_tab(pred_ols, "OLS", spec , actuals) ], axis=0)
result_table = pd.concat( [ result_table , forecast_table(pred_ols, "OLS", spec , actuals, dates) ], axis=0)

We observe the results of Nowcasting with OLS in different information windows.

In [ ]:
result_table

In [ ]:
# The bar every model has to clear: always predict the training-period mean of GDP growth
naive_nowcast = float(np.sqrt(np.mean((np.array(actuals) - naive_pred) ** 2)))
print(f"Training mean, RMSE: {naive_nowcast:.4f}")
performance_table

In [ ]:
result_table.plot()